# Production Deployment & Observability

This notebook accompanies the [Agent Foundry](https://agent-foundry-theta.vercel.app) LangChain roadmap.

You will learn how to add LangSmith tracing, monitoring middleware, cost tracking, and OpenTelemetry integration to production agents.

## 1. Install Dependencies

In [ ]:
!pip install -q langchain "langchain[openai]" langgraph langchain-mcp-adapters

## 2. Set Up Your API Keys

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
os.environ["LANGSMITH_API_KEY"] = getpass("Enter your LangSmith API key (or press Enter to skip): ")

## 3. Enable LangSmith Tracing

Set `LANGSMITH_TRACING=true` to capture every LLM call, tool invocation, and agent step.

In [ ]:
os.environ["LANGSMITH_TRACING"] = "true"

from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage

model = init_chat_model("gpt-4o-mini", model_provider="openai")

agent = create_react_agent(
    model=model,
    tools=[],
    prompt="You are a helpful assistant.",
)

result = agent.invoke({
    "messages": [HumanMessage(content="What is LangSmith?")]
})
print(result["messages"][-1].content)

## 4. Trace Metadata

Add metadata to traces for filtering and grouping in the LangSmith dashboard.

In [ ]:
result = agent.invoke(
    {"messages": [HumanMessage(content="Explain RAG in production")]},
    config={
        "metadata": {
            "user_id": "user-123",
            "session_id": "session-abc",
            "environment": "production",
        },
        "run_name": "rag-explanation",
    },
)
print(result["messages"][-1].content)

## 5. Monitoring Middleware

Track request counts, latency, and errors with custom middleware.

In [ ]:
import time
from langgraph.prebuilt.middleware import AgentMiddleware

class MonitoringMiddleware(AgentMiddleware):
    def __init__(self):
        self.metrics = {
            "total_requests": 0,
            "total_latency_ms": 0,
            "request_latencies": [],
        }

    def before_agent(self, state):
        state["_request_start"] = time.time()
        self.metrics["total_requests"] += 1
        return state

    def after_agent(self, state):
        elapsed_ms = (time.time() - state.get("_request_start", time.time())) * 1000
        self.metrics["total_latency_ms"] += elapsed_ms
        self.metrics["request_latencies"].append(round(elapsed_ms, 1))
        return state

monitor = MonitoringMiddleware()

agent_monitored = create_react_agent(
    model=model,
    tools=[],
    prompt="You are a helpful assistant.",
    middleware=[monitor],
)

for q in ["What is Python?", "What is JavaScript?", "What is Rust?"]:
    agent_monitored.invoke({"messages": [HumanMessage(content=q)]})

print(f"Total requests: {monitor.metrics['total_requests']}")
print(f"Latencies (ms): {monitor.metrics['request_latencies']}")
avg = monitor.metrics['total_latency_ms'] / monitor.metrics['total_requests']
print(f"Avg latency: {avg:.1f}ms")

## 6. Cost Tracking Middleware

Monitor per-request and cumulative LLM spending.

In [ ]:
class CostTrackingMiddleware(AgentMiddleware):
    COST_PER_1K_INPUT = 0.00015
    COST_PER_1K_OUTPUT = 0.0006

    def __init__(self):
        self.total_cost = 0.0
        self.request_costs = []

    def after_model(self, state):
        usage = state.get("_token_usage", {})
        input_cost = (usage.get("input_tokens", 0) / 1000) * self.COST_PER_1K_INPUT
        output_cost = (usage.get("output_tokens", 0) / 1000) * self.COST_PER_1K_OUTPUT
        request_cost = input_cost + output_cost
        self.total_cost += request_cost
        self.request_costs.append(request_cost)
        return state

cost_tracker = CostTrackingMiddleware()
print(f"Cost tracking middleware initialized.")
print(f"Input cost: ${CostTrackingMiddleware.COST_PER_1K_INPUT}/1K tokens")
print(f"Output cost: ${CostTrackingMiddleware.COST_PER_1K_OUTPUT}/1K tokens")

## 7. OpenTelemetry Integration

Export traces to any OTEL-compatible backend (Datadog, Honeycomb, Jaeger).

In [ ]:
otel_setup = """
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanExporter
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter

provider = TracerProvider()
exporter = OTLPSpanExporter(endpoint="http://localhost:4317")
provider.add_span_processor(BatchSpanExporter(exporter))
trace.set_tracer_provider(provider)
"""

print("OpenTelemetry setup (requires running OTEL collector):")
print(otel_setup)

## 8. A2A Protocol Overview

The Agent-to-Agent protocol enables cross-framework agent communication over HTTP.

In [ ]:
agent_card = {
    "name": "research-agent",
    "description": "Researches topics and produces summaries",
    "url": "https://my-agent.example.com",
    "capabilities": ["research", "summarization"],
    "protocol": "a2a/v1",
}

print("Agent Card (A2A):")
for key, value in agent_card.items():
    print(f"  {key}: {value}")

## 9. Combined Production Setup

Stack monitoring and logging middleware for a production-ready agent.

In [ ]:
prod_monitor = MonitoringMiddleware()

production_agent = create_react_agent(
    model=model,
    tools=[],
    prompt="You are a production assistant.",
    middleware=[prod_monitor],
)

questions = [
    "What is observability?",
    "How does tracing help in production?",
    "What metrics should I track for LLM apps?",
]

for q in questions:
    result = production_agent.invoke({"messages": [HumanMessage(content=q)]})
    print(f"Q: {q}")
    print(f"A: {result['messages'][-1].content[:100]}...\n")

print(f"\n--- Production Metrics ---")
print(f"Requests: {prod_monitor.metrics['total_requests']}")
print(f"Latencies: {prod_monitor.metrics['request_latencies']}")

## Summary

- Enable LangSmith tracing with `LANGSMITH_TRACING=true` — no code changes required
- Add metadata to traces for filtering by user, session, and environment
- Monitoring middleware tracks latency, request counts, and errors
- Cost tracking middleware monitors per-request LLM spending
- OpenTelemetry integration exports traces to Datadog, Honeycomb, Jaeger
- A2A protocol enables cross-framework agent communication over HTTP